# 02. 손실함수와 최적화 (Loss Functions & Optimization)

## 학습 목표
- 다양한 손실함수의 의미와 적절한 사용처 이해
- Cross-Entropy를 정보이론 관점에서 이해
- Gradient Descent 변형들의 차이 시각화
- SGD, Momentum, RMSProp, Adam을 직접 구현 (from scratch)
- Learning Rate Scheduling 비교

## 참고 자료
- [3Blue1Brown - Neural Networks (Gradient Descent)](https://www.youtube.com/watch?v=IHZwWFHWa-w)
- [Ruder - An overview of gradient descent optimization algorithms](https://ruder.io/optimizing-gradient-descent/)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from matplotlib.colors import LogNorm

np.random.seed(42)
torch.manual_seed(42)

## 1. 손실함수 (Loss Functions)

손실함수는 모델의 예측이 **얼마나 틀렸는지** 측정하는 함수.

학습 = 손실함수를 **최소화**하는 파라미터를 찾는 과정.

$$\theta^* = \arg\min_\theta \mathcal{L}(\theta)$$

### 1.1 회귀용 손실함수: MSE vs MAE

| 손실함수 | 수식 | 특성 |
|----------|------|------|
| MSE (Mean Squared Error) | $\frac{1}{n}\sum(y - \hat{y})^2$ | 큰 오차에 큰 페널티, 미분 가능 |
| MAE (Mean Absolute Error) | $\frac{1}{n}\sum|y - \hat{y}|$ | 이상치에 강건, 0에서 미분 불가 |

In [ ]:
# MSE vs MAE 비교
errors = np.linspace(-3, 3, 200)

mse_values = errors ** 2
mae_values = np.abs(errors)
huber_values = np.where(np.abs(errors) <= 1, 0.5 * errors ** 2, np.abs(errors) - 0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 손실 값 비교
ax = axes[0]
ax.plot(errors, mse_values, 'b-', linewidth=2, label='MSE: $(y - \\hat{y})^2$')
ax.plot(errors, mae_values, 'r-', linewidth=2, label='MAE: $|y - \\hat{y}|$')
ax.plot(errors, huber_values, 'g--', linewidth=2, label='Huber (delta=1)')
ax.set_xlabel('Error $(y - \\hat{y})$')
ax.set_ylabel('Loss')
ax.set_title('Loss 비교: MSE vs MAE vs Huber')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: 기울기(gradient) 비교
ax = axes[1]
mse_grad = 2 * errors  # d/de (e^2) = 2e
mae_grad = np.sign(errors)  # d/de |e| = sign(e)

ax.plot(errors, mse_grad, 'b-', linewidth=2, label='MSE gradient: $2(y - \\hat{y})$')
ax.plot(errors, mae_grad, 'r-', linewidth=2, label='MAE gradient: $\\text{sign}(y - \\hat{y})$')
ax.set_xlabel('Error')
ax.set_ylabel('Gradient')
ax.set_title('Gradient 비교')
ax.legend()
ax.grid(True, alpha=0.3)
ax.annotate('MSE: 오차가 클수록\ngradient도 큼', xy=(2, 4), fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.annotate('MAE: gradient가\n항상 일정', xy=(-2.5, -1.3), fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

In [ ]:
# 이상치(Outlier)에 대한 MSE vs MAE 비교
np.random.seed(42)
X = np.linspace(0, 10, 20).reshape(-1, 1)
y = 2 * X.ravel() + 3 + np.random.randn(20) * 1

# 이상치 추가
y_outlier = y.copy()
y_outlier[18] = 50  # 큰 이상치

# PyTorch로 MSE vs MAE 학습
X_t = torch.tensor(X, dtype=torch.float32)
y_t_clean = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
y_t_outlier = torch.tensor(y_outlier.reshape(-1, 1), dtype=torch.float32)

results = {}
for loss_name, criterion in [('MSE', nn.MSELoss()), ('MAE', nn.L1Loss())]:
    model = nn.Linear(1, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005)
    
    for epoch in range(500):
        pred = model(X_t)
        loss = criterion(pred, y_t_outlier)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    results[loss_name] = (model.weight.item(), model.bias.item())

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X, y_outlier, s=40, alpha=0.7, label='Data (with outlier)', zorder=5)
ax.scatter(X[18], y_outlier[18], s=100, c='red', marker='x', linewidths=3, label='Outlier', zorder=6)

X_line = np.linspace(0, 10, 100)
for name, (w, b) in results.items():
    ax.plot(X_line, w * X_line + b, linewidth=2, label=f'{name}: y={w:.2f}x + {b:.2f}')

# 이상치 없는 진짜 직선
ax.plot(X_line, 2 * X_line + 3, 'k--', alpha=0.5, label='True: y=2x+3')

ax.set_xlabel('X')
ax.set_ylabel('y')
ax.set_title('이상치가 있을 때: MSE는 크게 영향받고, MAE는 상대적으로 강건')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Cross-Entropy 깊이 파기

### 2.1 정보이론 배경

**정보량 (Information)**: 놀라움의 정도. 드문 사건일수록 정보량이 크다.

$$I(x) = -\log P(x)$$

**엔트로피 (Entropy)**: 평균 정보량. 불확실성의 척도.

$$H(P) = -\sum_x P(x) \log P(x)$$

**Cross-Entropy**: 실제 분포 $P$를 모델 분포 $Q$로 설명할 때의 평균 정보량.

$$H(P, Q) = -\sum_x P(x) \log Q(x)$$

**KL Divergence**: 두 분포의 차이.

$$D_{KL}(P \| Q) = H(P, Q) - H(P) \geq 0$$

→ Cross-Entropy를 최소화하면 KL Divergence도 최소화 (H(P)는 상수).
→ 즉, 모델의 분포를 실제 분포에 가깝게 만드는 것.

In [ ]:
# 정보이론 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 왼쪽: 정보량
ax = axes[0]
p = np.linspace(0.01, 1, 100)
info = -np.log2(p)
ax.plot(p, info, 'b-', linewidth=2)
ax.set_xlabel('P(x) - 사건의 확률')
ax.set_ylabel('$I(x) = -\\log_2 P(x)$ (bits)')
ax.set_title('정보량: 드문 사건일수록 정보량이 큼')
ax.annotate('동전 앞면 (P=0.5)\n→ 1 bit', xy=(0.5, 1), xytext=(0.6, 3),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.annotate('확실한 사건 (P=1)\n→ 0 bit', xy=(1, 0), xytext=(0.7, 1.5),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.grid(True, alpha=0.3)

# 중앙: 엔트로피 (이진 분류)
ax = axes[1]
p = np.linspace(0.01, 0.99, 100)
entropy = -p * np.log2(p) - (1-p) * np.log2(1-p)
ax.plot(p, entropy, 'b-', linewidth=2)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('P(Class 1)')
ax.set_ylabel('Entropy (bits)')
ax.set_title('Binary Entropy\nP=0.5일 때 최대 (가장 불확실)')
ax.grid(True, alpha=0.3)

# 오른쪽: Cross-Entropy 시각화
ax = axes[2]
# 실제 정답이 1일 때, 모델이 예측한 확률에 따른 loss
q = np.linspace(0.01, 0.99, 100)
ce_y1 = -np.log(q)         # y=1일 때 CE
ce_y0 = -np.log(1 - q)     # y=0일 때 CE

ax.plot(q, ce_y1, 'b-', linewidth=2, label='y=1: $-\\log(\\hat{y})$')
ax.plot(q, ce_y0, 'r-', linewidth=2, label='y=0: $-\\log(1-\\hat{y})$')
ax.set_xlabel('Model prediction $\\hat{y}$')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Binary Cross-Entropy\n정답과 멀수록 loss 급증')
ax.legend()
ax.set_ylim(0, 5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.2 Softmax + Cross-Entropy

다중 분류에서는 **Softmax**로 각 클래스의 확률을 계산하고, **Cross-Entropy**로 loss를 계산.

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

$$\mathcal{L} = -\sum_c y_c \log(\hat{y}_c)$$

PyTorch의 `nn.CrossEntropyLoss`는 **softmax + cross-entropy를 합쳐놓은 것**.

In [ ]:
def softmax(z):
    """Numerically stable softmax"""
    z_shifted = z - np.max(z)  # overflow 방지
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum()

def cross_entropy(y_true, y_pred):
    """Cross-entropy loss (one-hot y_true)"""
    return -np.sum(y_true * np.log(y_pred + 1e-9))

# 예시: 3클래스 분류
logits = np.array([2.0, 1.0, 0.1])  # 모델의 raw 출력 (logits)
probs = softmax(logits)

print(f"Logits:       {logits}")
print(f"Softmax 확률: {probs}")
print(f"합계:         {probs.sum():.4f} (항상 1)")
print()

# 각 클래스가 정답일 때의 CE loss
for true_class in range(3):
    y_true = np.zeros(3)
    y_true[true_class] = 1  # one-hot
    loss = cross_entropy(y_true, probs)
    print(f"정답이 class {true_class}일 때: CE = {loss:.4f} (예측 확률: {probs[true_class]:.4f})")

print("\n→ 정답 클래스의 확률이 높을수록 loss가 작다")

In [ ]:
# PyTorch와 직접 구현 비교
logits_t = torch.tensor([2.0, 1.0, 0.1]).unsqueeze(0)  # (1, 3)
target = torch.tensor([0])  # 정답: class 0

# PyTorch CrossEntropyLoss (내부에서 softmax + CE 자동 계산)
loss_pt = nn.CrossEntropyLoss()(logits_t, target)

# 직접 계산
probs_manual = softmax(np.array([2.0, 1.0, 0.1]))
loss_manual = -np.log(probs_manual[0])

print(f"PyTorch CrossEntropyLoss: {loss_pt.item():.4f}")
print(f"직접 계산 (softmax+CE): {loss_manual:.4f}")
print(f"같은가? {np.isclose(loss_pt.item(), loss_manual)}")

print("\n주의: PyTorch CrossEntropyLoss는 logits를 입력으로 받는다 (softmax 적용 전!)")
print("→ nn.CrossEntropyLoss = nn.LogSoftmax + nn.NLLLoss")

---
## 3. Gradient Descent 변형

### 파라미터 업데이트 규칙

$$\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}(\theta)$$

| 방법 | 한 번에 사용하는 데이터 | 특성 |
|------|------------------------|------|
| Batch GD | 전체 데이터 | 안정적이지만 느림, 메모리 많이 사용 |
| Stochastic GD (SGD) | 1개 데이터 | 빠르지만 노이즈 심함 |
| Mini-batch GD | 일부 데이터 (32, 64, ...) | 둘의 절충, **가장 많이 사용** |

In [ ]:
# Gradient Descent 변형 시각화
# 간단한 2D 함수에서 최적화 과정 비교

# Rosenbrock-like function: f(w1, w2) = (1-w1)^2 + 10*(w2-w1^2)^2
def loss_fn(w1, w2):
    return (1 - w1)**2 + 10 * (w2 - w1**2)**2

def grad_fn(w1, w2):
    dl_dw1 = -2 * (1 - w1) + 10 * 2 * (w2 - w1**2) * (-2 * w1)
    dl_dw2 = 10 * 2 * (w2 - w1**2)
    return np.array([dl_dw1, dl_dw2])

# 데이터를 시뮬레이션하여 Batch, Mini-batch, SGD 비교
np.random.seed(42)
n_data = 100

def noisy_grad(w1, w2, noise_level=0.0):
    """노이즈가 추가된 그래디언트 (SGD/Mini-batch 시뮬레이션)"""
    g = grad_fn(w1, w2)
    return g + np.random.randn(2) * noise_level * np.linalg.norm(g)

# 세 가지 방법으로 최적화
methods = {
    'Batch GD (noise=0)': 0.0,
    'Mini-batch GD (noise=0.3)': 0.3,
    'SGD (noise=1.0)': 1.0
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Loss surface
w1_range = np.linspace(-2, 2, 200)
w2_range = np.linspace(-1, 3, 200)
W1, W2 = np.meshgrid(w1_range, w2_range)
L = loss_fn(W1, W2)

for ax, (name, noise) in zip(axes, methods.items()):
    ax.contour(W1, W2, L, levels=np.logspace(-1, 3, 20), cmap='viridis', norm=LogNorm())
    
    # 최적화 실행
    w = np.array([-1.5, 2.0])  # 시작점
    lr = 0.002
    path = [w.copy()]
    
    np.random.seed(42)
    for _ in range(300):
        g = noisy_grad(w[0], w[1], noise)
        w = w - lr * g
        path.append(w.copy())
    
    path = np.array(path)
    ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=2, linewidth=0.5, alpha=0.7)
    ax.plot(path[0, 0], path[0, 1], 'go', markersize=10, label='Start')
    ax.plot(path[-1, 0], path[-1, 1], 'r*', markersize=15, label='End')
    ax.plot(1, 1, 'bx', markersize=12, markeredgewidth=3, label='Optimal (1,1)')
    
    final_loss = loss_fn(path[-1, 0], path[-1, 1])
    ax.set_xlabel('$w_1$')
    ax.set_ylabel('$w_2$')
    ax.set_title(f'{name}\nFinal loss: {final_loss:.4f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 4. 옵티마이저 직접 구현 (From Scratch)

### 4.1 Vanilla SGD

$$\theta \leftarrow \theta - \eta \cdot g$$

가장 기본적인 업데이트. 단점: 진동이 심하고, saddle point에서 멈출 수 있음.

In [ ]:
class SGD:
    def __init__(self, lr=0.01):
        self.lr = lr
    
    def step(self, params, grads):
        return params - self.lr * grads

# 테스트: f(x) = x^2 (최소값: x=0)
x = 5.0
opt = SGD(lr=0.1)
history = [x]

for _ in range(50):
    grad = 2 * x  # f'(x) = 2x
    x = opt.step(x, grad)
    history.append(x)

print(f"SGD: 시작 x=5.0 → 최종 x={x:.6f}")

### 4.2 SGD with Momentum

관성(모멘텀)을 추가하여 이전 방향의 영향을 유지.

$$v_t = \beta \cdot v_{t-1} + g_t$$
$$\theta \leftarrow \theta - \eta \cdot v_t$$

공이 경사면을 굴러가듯이, 이전 방향으로 가속. 진동 감소, 수렴 속도 향상.

In [ ]:
class MomentumSGD:
    def __init__(self, lr=0.01, momentum=0.9):
        self.lr = lr
        self.momentum = momentum
        self.v = None
    
    def step(self, params, grads):
        if self.v is None:
            self.v = np.zeros_like(grads)
        self.v = self.momentum * self.v + grads
        return params - self.lr * self.v

### 4.3 RMSProp

각 파라미터마다 **adaptive learning rate**를 적용.

$$s_t = \beta \cdot s_{t-1} + (1 - \beta) \cdot g_t^2$$
$$\theta \leftarrow \theta - \frac{\eta}{\sqrt{s_t + \epsilon}} \cdot g_t$$

gradient가 큰 방향은 lr을 줄이고, 작은 방향은 lr을 키움.

In [ ]:
class RMSProp:
    def __init__(self, lr=0.01, beta=0.9, epsilon=1e-8):
        self.lr = lr
        self.beta = beta
        self.epsilon = epsilon
        self.s = None
    
    def step(self, params, grads):
        if self.s is None:
            self.s = np.zeros_like(grads)
        self.s = self.beta * self.s + (1 - self.beta) * grads ** 2
        return params - self.lr * grads / (np.sqrt(self.s) + self.epsilon)

### 4.4 Adam (Adaptive Moment Estimation)

Momentum + RMSProp의 결합. **가장 널리 사용되는 옵티마이저**.

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad \text{(1st moment: 평균)}$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad \text{(2nd moment: 분산)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad \text{(bias correction)}$$
$$\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

In [ ]:
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None  # 1st moment
        self.v = None  # 2nd moment
        self.t = 0     # timestep
    
    def step(self, params, grads):
        if self.m is None:
            self.m = np.zeros_like(grads)
            self.v = np.zeros_like(grads)
        
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2
        
        # Bias correction
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)

### 4.5 옵티마이저 비교 시각화

In [ ]:
# Beale function: 여러 옵티마이저 경로 비교
def beale(x, y):
    return ((1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2 + (2.625 - x + x*y**3)**2)

def beale_grad(x, y):
    dx = (2*(1.5 - x + x*y)*(-1 + y) + 2*(2.25 - x + x*y**2)*(-1 + y**2) + 
          2*(2.625 - x + x*y**3)*(-1 + y**3))
    dy = (2*(1.5 - x + x*y)*x + 2*(2.25 - x + x*y**2)*(2*x*y) + 
          2*(2.625 - x + x*y**3)*(3*x*y**2))
    return np.array([dx, dy])

# 최적화 실행
optimizers = {
    'SGD (lr=0.0001)': SGD(lr=0.0001),
    'Momentum (lr=0.0001)': MomentumSGD(lr=0.0001, momentum=0.9),
    'RMSProp (lr=0.01)': RMSProp(lr=0.01, beta=0.9),
    'Adam (lr=0.05)': Adam(lr=0.05),
}

paths = {}
start = np.array([-1.5, -0.5])
n_steps = 500

for name, opt in optimizers.items():
    w = start.copy()
    path = [w.copy()]
    for _ in range(n_steps):
        g = beale_grad(w[0], w[1])
        # gradient clipping
        g_norm = np.linalg.norm(g)
        if g_norm > 100:
            g = g * 100 / g_norm
        w = opt.step(w, g)
        path.append(w.copy())
    paths[name] = np.array(path)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 경로 비교
ax = axes[0]
x_range = np.linspace(-2, 4, 300)
y_range = np.linspace(-2, 2, 300)
XX, YY = np.meshgrid(x_range, y_range)
ZZ = beale(XX, YY)

ax.contour(XX, YY, ZZ, levels=np.logspace(0, 4, 30), cmap='viridis', norm=LogNorm())
colors = ['red', 'blue', 'green', 'orange']
for (name, path), color in zip(paths.items(), colors):
    ax.plot(path[:, 0], path[:, 1], '.-', color=color, markersize=1, linewidth=1, alpha=0.8, label=name)
ax.plot(3, 0.5, 'k*', markersize=20, label='Optimal (3, 0.5)')
ax.plot(start[0], start[1], 'ko', markersize=10)
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Optimizer Trajectories (Beale Function)')
ax.legend(fontsize=8)
ax.set_xlim(-2, 4)
ax.set_ylim(-2, 2)

# 오른쪽: Loss 수렴 비교
ax = axes[1]
for (name, path), color in zip(paths.items(), colors):
    losses = [beale(p[0], p[1]) for p in path]
    ax.semilogy(losses, color=color, linewidth=1.5, label=name)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Loss 수렴 비교')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 최종 위치와 loss 출력
print(f"{'Optimizer':<30} {'Final Position':>20} {'Final Loss':>12}")
print("-" * 65)
for name, path in paths.items():
    final = path[-1]
    final_loss = beale(final[0], final[1])
    print(f"{name:<30} ({final[0]:>7.4f}, {final[1]:>7.4f}) {final_loss:>12.6f}")
print(f"{'Optimal':<30} ({3:>7.4f}, {0.5:>7.4f}) {beale(3, 0.5):>12.6f}")

### 4.6 PyTorch 내장 옵티마이저와 비교

In [ ]:
# PyTorch 옵티마이저로 간단한 회귀 문제 풀기
np.random.seed(42)
torch.manual_seed(42)

X_data = torch.linspace(0, 10, 100).reshape(-1, 1)
y_data = 3 * X_data + 7 + torch.randn(100, 1) * 2

pt_optimizers = {
    'SGD': lambda params: torch.optim.SGD(params, lr=0.001),
    'SGD+Momentum': lambda params: torch.optim.SGD(params, lr=0.001, momentum=0.9),
    'RMSProp': lambda params: torch.optim.RMSprop(params, lr=0.001),
    'Adam': lambda params: torch.optim.Adam(params, lr=0.01),
}

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['red', 'blue', 'green', 'orange']

for (name, opt_fn), color in zip(pt_optimizers.items(), colors):
    torch.manual_seed(42)
    model = nn.Linear(1, 1)
    optimizer = opt_fn(model.parameters())
    criterion = nn.MSELoss()
    
    losses = []
    for epoch in range(200):
        pred = model(X_data)
        loss = criterion(pred, y_data)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    ax.plot(losses, color=color, linewidth=1.5, label=f'{name} (final: {losses[-1]:.2f})')

ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('PyTorch Optimizers on Linear Regression')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Learning Rate Scheduling

학습 초반에는 큰 learning rate로 빠르게 수렴하고,
후반에는 작은 learning rate로 세밀하게 조정.

| Scheduler | 방식 | 특성 |
|-----------|------|------|
| StepLR | 일정 epoch마다 lr을 감소 | 단순하고 예측 가능 |
| CosineAnnealing | 코사인 곡선으로 lr 변화 | 부드러운 감소, warm restart 가능 |

In [ ]:
# Learning Rate Schedule 비교
epochs = 100

schedulers_config = {
    'Constant (lr=0.1)': lambda opt: None,
    'StepLR (step=30, gamma=0.1)': lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.1),
    'ExponentialLR (gamma=0.95)': lambda opt: torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.95),
    'CosineAnnealing (T_max=100)': lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: LR Schedule 시각화
ax = axes[0]
colors = ['gray', 'red', 'blue', 'green']

for (name, sched_fn), color in zip(schedulers_config.items(), colors):
    model_temp = nn.Linear(1, 1)
    opt = torch.optim.SGD(model_temp.parameters(), lr=0.1)
    scheduler = sched_fn(opt)
    
    lrs = []
    for epoch in range(epochs):
        lrs.append(opt.param_groups[0]['lr'])
        if scheduler is not None:
            scheduler.step()
    
    ax.plot(lrs, color=color, linewidth=2, label=name)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 오른쪽: Scheduler 적용 시 학습 곡선 비교
ax = axes[1]
X_data = torch.linspace(0, 10, 100).reshape(-1, 1)
y_data = 3 * X_data + 7 + torch.randn(100, 1) * 2

sched_for_training = {
    'No Schedule': None,
    'StepLR': lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.1),
    'CosineAnnealing': lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100),
}

for (name, sched_fn), color in zip(sched_for_training.items(), ['gray', 'red', 'green']):
    torch.manual_seed(42)
    model_temp = nn.Linear(1, 1)
    opt = torch.optim.SGD(model_temp.parameters(), lr=0.01)
    scheduler = sched_fn(opt) if sched_fn else None
    criterion = nn.MSELoss()
    
    losses = []
    for epoch in range(200):
        pred = model_temp(X_data)
        loss = criterion(pred, y_data)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if scheduler:
            scheduler.step()
        losses.append(loss.item())
    
    ax.plot(losses, color=color, linewidth=1.5, label=f'{name} (final: {losses[-1]:.2f})')

ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Learning Rate Schedule의 효과')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Adam 옵티마이저를 from scratch로 구현하고 PyTorch와 비교

위에서 구현한 Adam 클래스를 사용하여 간단한 회귀 문제를 풀고,
PyTorch의 `torch.optim.Adam`과 결과를 비교하세요.

In [ ]:
# 데이터: y = 3x + 7 + noise
np.random.seed(42)
X_ex = np.linspace(0, 10, 50).reshape(-1, 1)
y_ex = 3 * X_ex.ravel() + 7 + np.random.randn(50) * 2

# TODO: 직접 구현한 Adam으로 선형 회귀
#   1. w, b를 랜덤 초기화
#   2. 200 step 동안:
#      - 예측: y_hat = w * X + b
#      - MSE loss 계산
#      - gradient 계산: dL/dw = -2/n * X^T(y - y_hat), dL/db = -2/n * sum(y - y_hat)
#      - Adam으로 w, b 업데이트
#   3. PyTorch Adam의 결과와 비교 (loss 곡선 그래프)


### 연습 2: Learning Rate의 영향 실험

learning rate를 너무 크게/적당히/너무 작게 설정했을 때의 학습 곡선을 비교하세요.

In [ ]:
# TODO: lr = [0.5, 0.01, 0.0001]로 SGD 학습
#   - nn.Linear(1, 1) 모델
#   - 각 lr에 대해 300 epoch 학습
#   - 3개의 loss 곡선을 하나의 그래프에 시각화
#   - lr이 너무 크면 발산, 너무 작으면 수렴 느림을 확인
#   - (보너스) lr=0.5에서 발산하면 gradient clipping을 적용해보세요


---
## 핵심 정리

| 개념 | 핵심 내용 | ML/DL에서의 역할 |
|------|-----------|-------------------|
| MSE | $(y - \hat{y})^2$ 평균 | 회귀의 기본 손실함수 |
| MAE | $|y - \hat{y}|$ 평균 | 이상치에 강건한 회귀 손실 |
| Cross-Entropy | $-y\log\hat{y}$ | 분류의 기본 손실함수, 정보이론 기반 |
| Softmax | $e^{z_i} / \sum e^{z_j}$ | 다중 분류에서 확률 계산 |
| SGD | $\theta - \eta g$ | 가장 기본적인 최적화 |
| Momentum | 이전 방향 가속 | SGD의 진동 감소 |
| RMSProp | adaptive learning rate | 파라미터별 학습률 조절 |
| Adam | Momentum + RMSProp | 가장 널리 사용되는 옵티마이저 |
| LR Scheduling | 학습 중 lr 감소 | 수렴 안정성 향상 |

**다음 노트북**: [03-evaluation-metrics.ipynb](03-evaluation-metrics.ipynb) - 평가 지표